In [1]:
import cv2
import mediapipe as mp
import numpy as np
import math
import time

from mediapipe import solutions
mp_hands = solutions.hands
mp_drawing = solutions.drawing_utils

def get_distance(a, b):
    return math.hypot(a.x - b.x, a.y - b.y)

def get_gesture_name(finger_states):
    gesture_map = {
        '[1, 1, 0, 0, 1]': "I Love You",
        '[0, 1, 1, 1, 1]': "Hello (Static)",
        '[0, 0, 0, 0, 0]': "Sorry (Static)",
        '[0, 1, 1, 0, 0]': "Thank You (Static)",
        '[0, 1, 1, 0, 1]': "OK",
        '[0, 1, 1, 1, 0]': "Peace",
        '[1, 0, 0, 0, 0]': "Thumbs Up",
        '[1, 1, 1, 1, 1]': "Stop",
        '[1, 0, 0, 0, 1]': "Rock On",
        '[0, 0, 1, 0, 0]':  "Fuck you bitch",
    
    }
    return gesture_map.get(str(finger_states), "Unknown")

def analyze_hand(hand_landmarks, handedness, prev_centers, motion_buffers, motion_times):
    landmarks = hand_landmarks.landmark
    hand_id = handedness.classification[0].index  # 0 for left, 1 for right
    
    # Get hand label (Left/Right)
    hand_label = handedness.classification[0].label
    
    center_x = np.mean([lm.x for lm in landmarks])
    center_y = np.mean([lm.y for lm in landmarks])
    curr_center = (center_x, center_y)
    
    # Get finger states
    finger_tips = [4, 8, 12, 16, 20]
    finger_states = []
    
    # For each finger (thumb, index, middle, ring, pinky)
    for i in range(5):
        tip_idx = finger_tips[i]
        pip_idx = tip_idx - 2
        
        if i == 0:  # Thumb - different logic
            # For thumb, check if it's extended based on hand orientation
            if hand_label == "Left":
                # For left hand, thumb points right when extended
                finger_states.append(1 if landmarks[tip_idx].x > landmarks[pip_idx].x else 0)
            else:
                # For right hand, thumb points left when extended
                finger_states.append(1 if landmarks[tip_idx].x < landmarks[pip_idx].x else 0)
        else:  # Other fingers
            finger_states.append(1 if landmarks[tip_idx].y < landmarks[pip_idx].y else 0)
    
    static_gesture = get_gesture_name(finger_states)
    
    # Initialize motion tracking for this hand if not exists
    if hand_id not in motion_buffers:
        motion_buffers[hand_id] = []
        motion_times[hand_id] = time.time()
        prev_centers[hand_id] = curr_center
    
    # Detect dynamic movement
    gesture = static_gesture
    if prev_centers[hand_id] is not None:
        prev_x, prev_y = prev_centers[hand_id]
        dx, dy = curr_center[0] - prev_x, curr_center[1] - prev_y
        motion_buffers[hand_id].append((dx, dy))
        
        if len(motion_buffers[hand_id]) > 5:
            motion_buffers[hand_id].pop(0)
        
        # Check for fast movement
        if len(motion_buffers[hand_id]) >= 3:
            avg_dx = np.mean([m[0] for m in motion_buffers[hand_id]])
            avg_dy = np.mean([m[1] for m in motion_buffers[hand_id]])
            fast_move = abs(avg_dx) > 0.02 or abs(avg_dy) > 0.02
            
            if fast_move and time.time() - motion_times[hand_id] > 1.0:
                if abs(avg_dx) > abs(avg_dy):
                    gesture = f"{hand_label} Hello (Waving)"
                elif avg_dy < -0.02:
                    gesture = f"{hand_label} Thank You (Outward)"
                elif avg_dy > 0.02:
                    gesture = f"{hand_label} Sorry (Motion)"
                motion_times[hand_id] = time.time()
    
    prev_centers[hand_id] = curr_center
    confidence = round(sum(finger_states)/5 * 100, 1)
    
    return {
        'hand_label': hand_label,
        'gesture': gesture,
        'confidence': confidence,
        'center': curr_center,
        'landmarks': landmarks,
        'hand_id': hand_id
    }

cap = cv2.VideoCapture(0)
if not cap.isOpened():
    print("Error: Could not access camera.")
    exit()

# Track each hand separately
prev_centers = {}
motion_buffers = {}
motion_times = {}

# Increase max_num_hands to 2
with mp_hands.Hands(max_num_hands=2, min_detection_confidence=0.8, min_tracking_confidence=0.8) as hands:
    while True:
        ret, frame = cap.read()
        if not ret:
            print("Camera error: frame not captured.")
            break
        
        frame = cv2.flip(frame, 1)
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        result = hands.process(rgb)
        
        # Initialize display variables
        gesture_text = "No Hands Detected"
        combined_confidence = 0.0
        hand_results = []
        
        if result.multi_hand_landmarks:
            hand_results = []
            num_hands = len(result.multi_hand_landmarks)
            
            for hand_idx, hand_landmarks in enumerate(result.multi_hand_landmarks):
                # Get handedness (left/right)
                handedness = result.multi_handedness[hand_idx]
                
                # Analyze this hand
                hand_info = analyze_hand(
                    hand_landmarks, 
                    handedness, 
                    prev_centers, 
                    motion_buffers, 
                    motion_times
                )
                
                hand_results.append(hand_info)
                
                # Draw landmarks and connections
                mp_drawing.draw_landmarks(
                    frame, 
                    hand_landmarks, 
                    mp_hands.HAND_CONNECTIONS
                )
                
                # Draw hand label near the wrist
                wrist = hand_info['landmarks'][0]
                h, w, _ = frame.shape
                wrist_x, wrist_y = int(wrist.x * w), int(wrist.y * h)
                
                cv2.putText(frame, 
                          hand_info['hand_label'], 
                          (wrist_x - 30, wrist_y - 20),
                          cv2.FONT_HERSHEY_SIMPLEX, 
                          0.7, 
                          (0, 255, 255), 
                          2)
            
            # Combine results for display
            if hand_results:
                gestures = [hr['gesture'] for hr in hand_results]
                confidences = [hr['confidence'] for hr in hand_results]
                
                if len(hand_results) == 1:
                    gesture_text = f"{hand_results[0]['hand_label']}: {hand_results[0]['gesture']}"
                    combined_confidence = hand_results[0]['confidence']
                else:
                    # Sort by hand position (left to right) for consistent display
                    hand_results.sort(key=lambda x: x['center'][0])
                    gesture_text = " & ".join([f"{hr['hand_label']}: {hr['gesture']}" for hr in hand_results])
                    combined_confidence = np.mean(confidences)
        
        # Display information
        y_offset = 40
        cv2.putText(frame, f"Hands: {len(hand_results)}", 
                   (10, y_offset), 
                   cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        y_offset += 40
        
        cv2.putText(frame, f"Gesture: {gesture_text}", 
                   (10, y_offset), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
        y_offset += 40
        
        if hand_results:
            cv2.putText(frame, f"Avg Confidence: {combined_confidence:.1f}%", 
                       (10, y_offset), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 0), 2)
        
        cv2.imshow("Dual-Hand Sign Language Detection", frame)
        
        if cv2.waitKey(1) & 0xFF == 27:  # ESC to exit
            break

cap.release()
cv2.destroyAllWindows()

Camera error: frame not captured.
